# <font color='lightgreen'>03 Build Star Schema</font> ⭐
##### Construcción del esquema estrella (dimensiones y hechos) a partir de datos limpios.
---

### <font color='lightgreen'>Librerías</font>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("✅ Librerías cargadas correctamente")
print('Versión de pandas:', pd.__version__)
print('Versión de numpy:', np.__version__)

✅ Librerías cargadas correctamente
Versión de pandas: 3.0.1
Versión de numpy: 2.4.3


### <font color='lightgreen'>Configuración</font>

In [2]:
# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

# Rutas
DATA_CLEAN = "../data/clean"
DATA_CURATED = "../data/curated"

# Crear carpeta de salida
Path(DATA_CURATED).mkdir(parents=True, exist_ok=True)

# Diccionario para dimensiones y hechos
dimensiones = {}
hechos = {}

print(f"📁 Datos limpios: {DATA_CLEAN}")
print(f"📁 Modelo estrella: {DATA_CURATED}")

📁 Datos limpios: ../data/clean
📁 Modelo estrella: ../data/curated


### <font color="lightgreen">1. Cargar datos limpios</font>

In [3]:
print("\n" + "="*50)
print("CARGANDO DATOS LIMPIOS...")
print("="*50)

datos = {}
archivos_config = [
    ("compras", ["fecha_emision", "fecha_pago"]),
    ("consumo_energetico", ["fecha"]),
    ("encuestas", ["fecha_encuesta"]),
    ("eventos_rrhh", ["fecha"]),
    ("personal_nomina", ["mes", "fecha_ingreso"]),
    ("residuos", ["fecha_semana"]),
    ("ventas", ["fecha"])
]

for nombre, columnas_fecha in archivos_config:
    file_path = f"{DATA_CLEAN}/{nombre}_clean.csv"
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        for col in columnas_fecha:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        datos[nombre] = df
        print(f"✅ {nombre}: {len(df):,} filas, {len(df.columns)} columnas")
        for col in columnas_fecha:
            if col in df.columns:
                print(f"   → {col}: convertida a datetime")
    except Exception as e:
        print(f"❌ {nombre}: {e}")


CARGANDO DATOS LIMPIOS...
✅ compras: 864 filas, 8 columnas
   → fecha_emision: convertida a datetime
   → fecha_pago: convertida a datetime
✅ consumo_energetico: 178 filas, 13 columnas
   → fecha: convertida a datetime
✅ encuestas: 189 filas, 12 columnas
   → fecha_encuesta: convertida a datetime
✅ eventos_rrhh: 537 filas, 10 columnas
   → fecha: convertida a datetime
✅ personal_nomina: 1,378 filas, 12 columnas
   → mes: convertida a datetime
   → fecha_ingreso: convertida a datetime
✅ residuos: 2,508 filas, 11 columnas
   → fecha_semana: convertida a datetime
✅ ventas: 43,893 filas, 15 columnas
   → fecha: convertida a datetime


### <font color="lightgreen">2. Crear dimensiones</font>

#### <font color="lightgreen">2.1 Dimensión Tiempo</font>

In [4]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN TIEMPO...")
print("="*50)

todas_fechas = []
if 'fecha_emision' in datos['compras'].columns:
    todas_fechas.extend(datos['compras']['fecha_emision'].dropna().tolist())
if 'fecha' in datos['consumo_energetico'].columns:
    todas_fechas.extend(datos['consumo_energetico']['fecha'].dropna().tolist())
if 'fecha_encuesta' in datos['encuestas'].columns:
    todas_fechas.extend(datos['encuestas']['fecha_encuesta'].dropna().tolist())
if 'fecha' in datos['eventos_rrhh'].columns:
    todas_fechas.extend(datos['eventos_rrhh']['fecha'].dropna().tolist())
if 'mes' in datos['personal_nomina'].columns:
    todas_fechas.extend(datos['personal_nomina']['mes'].dropna().tolist())
if 'fecha_semana' in datos['residuos'].columns:
    todas_fechas.extend(datos['residuos']['fecha_semana'].dropna().tolist())
if 'fecha' in datos['ventas'].columns:
    todas_fechas.extend(datos['ventas']['fecha'].dropna().tolist())

fechas_unicas = sorted(set(todas_fechas))
dim_tiempo = pd.DataFrame({'fecha': fechas_unicas})
dim_tiempo['id_tiempo'] = dim_tiempo['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_tiempo['anio'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.quarter
dim_tiempo['semana'] = dim_tiempo['fecha'].dt.isocalendar().week
dim_tiempo['nombre_mes'] = dim_tiempo['fecha'].dt.strftime('%B')
dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.day_name()
dim_tiempo['es_finde'] = (dim_tiempo['fecha'].dt.dayofweek >= 5).astype(int)
dim_tiempo = dim_tiempo[['id_tiempo', 'fecha', 'anio', 'mes', 'dia', 'trimestre',
                        'semana', 'nombre_mes', 'dia_semana', 'es_finde']]

print(f"✅ dim_tiempo: {len(dim_tiempo)} fechas únicas")
if len(dim_tiempo) > 0:
    print(f"   📅 Rango: {dim_tiempo['fecha'].min().date()} → {dim_tiempo['fecha'].max().date()}")
    print(f"   📊 Años: {sorted(dim_tiempo['anio'].unique())}")
dimensiones['dim_tiempo'] = dim_tiempo



CREANDO DIMENSIÓN TIEMPO...
✅ dim_tiempo: 2567 fechas únicas
   📅 Rango: 2018-01-01 → 2025-12-31
   📊 Años: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


#### <font color="lightgreen">2.2 Dimensión Area</font>

In [5]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN ÁREA...")
print("="*50)

areas = set()
if 'area' in datos['personal_nomina'].columns:
    areas.update(datos['personal_nomina']['area'].dropna().unique())
if 'area' in datos['encuestas'].columns:
    areas.update(datos['encuestas']['area'].dropna().unique())
if 'area' in datos['eventos_rrhh'].columns:
    areas.update(datos['eventos_rrhh']['area'].dropna().unique())

dim_area = pd.DataFrame(sorted(areas), columns=['nombre_area'])
dim_area.insert(0, 'id_area', range(1, len(dim_area) + 1))
print(f"✅ dim_area: {len(dim_area)} áreas únicas")
print(f"   📍 Áreas: {list(dim_area['nombre_area'])}")
dimensiones['dim_area'] = dim_area


CREANDO DIMENSIÓN ÁREA...
✅ dim_area: 5 áreas únicas
   📍 Áreas: ['Administración', 'Logística', 'Taller', 'Todas', 'Ventas']


#### <font color="lightgreen">2.3 Dimensión Empleado</font>

In [6]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN EMPLEADO...")
print("="*50)

if 'personal_nomina' in datos:
    df_nomina = datos['personal_nomina']
    columnas_empleado = ['id_empleado', 'nombre', 'area', 'genero', 'fecha_ingreso']
    columnas_existentes = [col for col in columnas_empleado if col in df_nomina.columns]
    dim_empleado = df_nomina[columnas_existentes].drop_duplicates(subset=['id_empleado']).copy()
    dim_empleado = dim_empleado.sort_values('id_empleado').reset_index(drop=True)
    print(f"✅ dim_empleado: {len(dim_empleado)} empleados únicos")
    print(f"   👥 Muestra: {dim_empleado.head(3).to_dict('records')}")
    dimensiones['dim_empleado'] = dim_empleado
else:
    print("⚠️  No se encontró datos de nómina para crear dim_empleado")


CREANDO DIMENSIÓN EMPLEADO...
✅ dim_empleado: 25 empleados únicos
   👥 Muestra: [{'id_empleado': 'E001', 'nombre': 'Fernández, Carlos', 'area': 'Administración', 'genero': 'M', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}, {'id_empleado': 'E002', 'nombre': 'López, María', 'area': 'Ventas', 'genero': 'F', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}, {'id_empleado': 'E003', 'nombre': 'García, Juan', 'area': 'Ventas', 'genero': 'M', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}]


#### <font color="lightgreen">2.4 Dimensión Proveedor</font>

In [7]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN PROVEEDOR...")
print("="*50)

if 'compras' in datos and 'proveedor' in datos['compras'].columns:
    dim_proveedor = pd.DataFrame(datos['compras']['proveedor'].dropna().unique(), columns=['nombre_proveedor'])
    dim_proveedor.insert(0, 'id_proveedor', range(1, len(dim_proveedor) + 1))
    print(f"✅ dim_proveedor: {len(dim_proveedor)} proveedores únicos")
    dimensiones['dim_proveedor'] = dim_proveedor
else:
    print("⚠️  Datos de proveedor no disponibles")


CREANDO DIMENSIÓN PROVEEDOR...
✅ dim_proveedor: 5 proveedores únicos


#### <font color="lightgreen">2.5 Dimensión Categoría</font>

In [8]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN CATEGORÍA...")
print("="*50)

if 'ventas' in datos and 'categoria' in datos['ventas'].columns:
    dim_categoria = pd.DataFrame(datos['ventas']['categoria'].dropna().unique(), columns=['nombre_categoria'])
    dim_categoria.insert(0, 'id_categoria', range(1, len(dim_categoria) + 1))
    print(f"✅ dim_categoria: {len(dim_categoria)} categorías únicas")
    print(f"   📦 Categorías: {list(dim_categoria['nombre_categoria'])}")
    dimensiones['dim_categoria'] = dim_categoria
else:
    print("⚠️  Datos de categorías no disponibles")



CREANDO DIMENSIÓN CATEGORÍA...
✅ dim_categoria: 10 categorías únicas
   📦 Categorías: ['Periféricos', 'Smartphones', 'Seguridad', 'TV/Audio', 'Accesorios', 'Computación', 'Tablets', 'Almacenamiento', 'Redes', 'Servicios']


#### <font color="lightgreen">2.6 Dimensión Métrica (ampliada)</font>

In [9]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN MÉTRICA (ampliada)...")
print("="*50)

metricas = [
    (1, 'Ingresos', 'Financiera', 'Ventas', 'ARS', 'Suma de ventas', 'Crítica', 'Mensual', 'SUM', 'UP', 'Ingresos totales por ventas'),
    (2, 'Costo Compras', 'Financiera', 'Insumos', 'ARS', 'Suma de compras', 'Importante', 'Mensual', 'SUM', 'DOWN', 'Costos de adquisición'),
    (3, 'Gasto Personal', 'Financiera', 'RRHH', 'ARS', 'Suma de nóminas', 'Importante', 'Mensual', 'SUM', 'DOWN', 'Salarios y bonificaciones'),
    (4, 'Consumo Energético', 'E', 'Energía', 'kWh', 'Suma de kWh consumidos', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Electricidad + gas (convertido)'),
    (5, 'Tasa Reciclaje', 'E', 'Residuos', '%', '(kg_reciclados/kg_generados)*100', 'Importante', 'Mensual', 'AVG', 'UP', 'Porcentaje de residuos reciclados'),
    (6, 'Satisfacción Laboral', 'S', 'Clima', 'puntos', 'Promedio encuestas', 'Importante', 'Trimestral', 'AVG', 'UP', 'Satisfacción general (1-10)'),
    (7, 'Horas Capacitación', 'S', 'Desarrollo', 'horas', 'Suma horas capacitación', 'Secundaria', 'Mensual', 'SUM', 'UP', 'Horas de formación'),
    (8, 'Días Baja Accidentes', 'S', 'Seguridad', 'días', 'Suma días baja por accidente', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Días perdidos por accidentes'),
    (9, 'Margen Neto', 'Financiera', 'Rentabilidad', '%', '(Ingresos - Costos - Gastos)/Ingresos * 100', 'Crítica', 'Mensual', 'AVG', 'UP', 'Rentabilidad final'),
    (10, 'Huella Carbono', 'E', 'Emisiones', 'tCO2e', 'Consumo_energético * factor_emision', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Emisiones estimadas (factor 0.4 kgCO2/kWh)'),
    (11, 'Intensidad Energética', 'E', 'Eficiencia', 'kWh/ARS', 'Consumo_energético / Ingresos', 'Importante', 'Mensual', 'AVG', 'DOWN', 'Eficiencia energética por peso'),
    (12, 'Productividad Empleado', 'Financiera', 'Productividad', 'ARS/empl', 'Ingresos / N° empleados', 'Importante', 'Mensual', 'AVG', 'UP', 'Ingreso por empleado'),
]
columnas_metrica = ['id_metrica', 'nombre', 'categoria', 'subcategoria', 'unidad', 'formula',
                    'prioridad', 'frecuencia_recomendada', 'tipo_agregacion', 'tendencia_deseada', 'descripcion']
dim_metrica = pd.DataFrame(metricas, columns=columnas_metrica)
print(f"✅ dim_metrica: {len(dim_metrica)} métricas definidas")
print(f"   📊 Categorías: {dim_metrica['categoria'].value_counts().to_dict()}")
print(f"   🎯 Tendencia: {dim_metrica['tendencia_deseada'].value_counts().to_dict()}")
dimensiones['dim_metrica'] = dim_metrica


CREANDO DIMENSIÓN MÉTRICA (ampliada)...
✅ dim_metrica: 12 métricas definidas
   📊 Categorías: {'Financiera': 5, 'E': 4, 'S': 3}
   🎯 Tendencia: {'UP': 6, 'DOWN': 6}


#### <font color="lightgreen">2.7 Dimensión Objetivo</font>

In [10]:
print("\n" + "="*50)
print("CREANDO DIMENSIÓN OBJETIVO...")
print("="*50)

objetivos = [
    (1, 2024, 100000000, 95000000, 85000000, 0.10),
    (2, 2024, 60000000, 65000000, 70000000, 0.05),
    (3, 2024, 20000000, 21000000, 23000000, 0.05),
    (4, 2024, 45000, 48000, 52000, 0.15),
    (5, 2024, 60, 50, 30, 0.10),
    (6, 2024, 7.5, 7.0, 6.0, 0.10),
    (7, 2024, 200, 150, 100, 0.05),
    (8, 2024, 10, 20, 30, 0.15),
    (9, 2024, 12, 10, 6, 0.15),
    (10, 2024, 18, 20, 25, 0.15),
    (11, 2024, 0.0005, 0.0006, 0.0008, 0.05),
    (12, 2024, 3000000, 2800000, 2500000, 0.05),
]
dim_objetivo = pd.DataFrame(objetivos, columns=['id_metrica', 'anio', 'valor_objetivo',
                                                'umbral_verde', 'umbral_amarillo', 'peso_relativo'])
dim_objetivo.insert(0, 'id_objetivo', range(1, len(dim_objetivo)+1))
print(f"✅ dim_objetivo: {len(dim_objetivo)} registros (metas anuales)")
print(f"   🎯 Años incluidos: {sorted(dim_objetivo['anio'].unique())}")
dimensiones['dim_objetivo'] = dim_objetivo


CREANDO DIMENSIÓN OBJETIVO...
✅ dim_objetivo: 12 registros (metas anuales)
   🎯 Años incluidos: [np.int64(2024)]


### <font color="lightgreen">3. Crear Tabla de Hechos (fact_monitoreo)</font>

In [11]:
mediciones = []
id_monitoreo_counter = 1
mapa_id_tiempo = dict(zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo']))

#### <font color="lightgreen">3.1 Ingresos</font>

In [12]:
if 'ventas' in datos and 'subtotal_ars' in datos['ventas'].columns:
    df = datos['ventas']
    ingresos = df.groupby(pd.Grouper(key='fecha', freq='ME'))['subtotal_ars'].sum().reset_index()
    for _, row in ingresos.iterrows():
        id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
        if id_t:
            mediciones.append([id_monitoreo_counter, id_t, 1, 1, row['subtotal_ars'], 'ventas'])
            id_monitoreo_counter += 1
    print(f"   ✅ Ingresos: {len(ingresos)} registros mensuales")

   ✅ Ingresos: 96 registros mensuales


#### <font color="lightgreen">3.2 Costo Compras</font>

In [13]:
if 'compras' in datos and 'monto_ars' in datos['compras'].columns:
    df = datos['compras']
    compras = df.groupby(pd.Grouper(key='fecha_emision', freq='ME'))['monto_ars'].sum().reset_index()
    for _, row in compras.iterrows():
        id_t = mapa_id_tiempo.get(row['fecha_emision'].replace(day=1))
        if id_t:
            mediciones.append([id_monitoreo_counter, id_t, 2, 1, row['monto_ars'], 'compras'])
            id_monitoreo_counter += 1
    print(f"   ✅ Costo Compras: {len(compras)} registros mensuales")


   ✅ Costo Compras: 85 registros mensuales


#### <font color="lightgreen">3.3 Gasto Personal</font>

In [14]:
if 'personal_nomina' in datos and 'total_devengado' in datos['personal_nomina'].columns:
    df = datos['personal_nomina']
    nomina = df.groupby(pd.Grouper(key='mes', freq='ME'))['total_devengado'].sum().reset_index()
    for _, row in nomina.iterrows():
        id_t = mapa_id_tiempo.get(row['mes'].replace(day=1))
        if id_t:
            mediciones.append([id_monitoreo_counter, id_t, 3, 1, row['total_devengado'], 'personal_nomina'])
            id_monitoreo_counter += 1
    print(f"   ✅ Gasto Personal: {len(nomina)} registros mensuales")

   ✅ Gasto Personal: 96 registros mensuales


#### <font color="lightgreen">3.4 Consumo Energético</font>

In [15]:
if 'consumo_energetico' in datos:
    df = datos['consumo_energetico']
    col_energia = 'kwh_totales' if 'kwh_totales' in df.columns else ('kwh_consumidos' if 'kwh_consumidos' in df.columns else None)
    if col_energia:
        energia = df.groupby(pd.Grouper(key='fecha', freq='ME'))[col_energia].sum().reset_index()
        for _, row in energia.iterrows():
            id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_t:
                mediciones.append([id_monitoreo_counter, id_t, 4, 1, row[col_energia], 'consumo_energetico'])
                id_monitoreo_counter += 1
        print(f"   ✅ Consumo Energético: {len(energia)} registros mensuales")

   ✅ Consumo Energético: 85 registros mensuales


#### <font color="lightgreen">3.5 Tasa Reciclaje</font>

In [16]:
if 'residuos' in datos and 'tasa_reciclaje' in datos['residuos'].columns:
    df = datos['residuos']
    reciclaje = df.groupby(pd.Grouper(key='fecha_semana', freq='ME'))['tasa_reciclaje'].mean().reset_index()
    for _, row in reciclaje.iterrows():
        id_t = mapa_id_tiempo.get(row['fecha_semana'].replace(day=1))
        if id_t:
            mediciones.append([id_monitoreo_counter, id_t, 5, 1, row['tasa_reciclaje'], 'residuos'])
            id_monitoreo_counter += 1
    print(f"   ✅ Tasa Reciclaje: {len(reciclaje)} registros mensuales")

   ✅ Tasa Reciclaje: 96 registros mensuales


#### <font color="lightgreen">3.6 Satisfacción Laboral</font>

In [17]:
if 'encuestas' in datos and 'satisfaccion_gral' in datos['encuestas'].columns:
    df = datos['encuestas']
    sat = df.groupby(pd.Grouper(key='fecha_encuesta', freq='QE'))['satisfaccion_gral'].mean().reset_index()
    for _, row in sat.iterrows():
        id_t = mapa_id_tiempo.get(row['fecha_encuesta'].replace(day=1))
        if id_t:
            mediciones.append([id_monitoreo_counter, id_t, 6, 1, row['satisfaccion_gral'], 'encuestas'])
            id_monitoreo_counter += 1
    print(f"   ✅ Satisfacción Laboral: {len(sat)} registros trimestrales")

   ✅ Satisfacción Laboral: 31 registros trimestrales

#### <font color="lightgreen">3.7 Horas Capacitación</font>

In [18]:
if 'eventos_rrhh' in datos and 'horas_capacitacion' in datos['eventos_rrhh'].columns:
    df = datos['eventos_rrhh']
    cap = df[df.get('tipo_evento', '') == 'Capacitación']
    if not cap.empty:
        horas = cap.groupby(pd.Grouper(key='fecha', freq='ME'))['horas_capacitacion'].sum().reset_index()
        for _, row in horas.iterrows():
            id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_t:
                mediciones.append([id_monitoreo_counter, id_t, 7, 1, row['horas_capacitacion'], 'eventos_rrhh'])
                id_monitoreo_counter += 1
        print(f"   ✅ Horas Capacitación: {len(horas)} registros mensuales")

   ✅ Horas Capacitación: 96 registros mensuales


#### <font color="lightgreen">3.8 Días Baja Accidentes</font>

In [19]:
if 'eventos_rrhh' in datos and 'dias_baja' in datos['eventos_rrhh'].columns:
    df = datos['eventos_rrhh']
    acc = df[df.get('tipo_evento', '') == 'Accidente Laboral']
    if not acc.empty:
        dias = acc.groupby(pd.Grouper(key='fecha', freq='ME'))['dias_baja'].sum().reset_index()
        for _, row in dias.iterrows():
            id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_t:
                mediciones.append([id_monitoreo_counter, id_t, 8, 1, row['dias_baja'], 'eventos_rrhh'])
                id_monitoreo_counter += 1
        print(f"   ✅ Días Baja Accidentes: {len(dias)} registros mensuales")

   ✅ Días Baja Accidentes: 96 registros mensuales


#### <font color="lightgreen">3.9 Métricas Derivadas</font>

In [20]:
if len(mediciones) > 0:
    # Convertir a DataFrame para pivote
    cols = ['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente']
    df_med = pd.DataFrame(mediciones, columns=cols)
    pivot = df_med.pivot_table(index='id_tiempo', columns='id_metrica', values='valor', aggfunc='first').reset_index()

    # Número de empleados por mes
    if 'personal_nomina' in datos:
        df_nom = datos['personal_nomina']
        empleados_mes = df_nom.groupby(pd.Grouper(key='mes', freq='ME'))['id_empleado'].nunique().reset_index()
        empleados_mes['id_tiempo'] = empleados_mes['mes'].dt.strftime('%Y%m%d').astype(int)
        empleados_mes = empleados_mes[['id_tiempo', 'id_empleado']].rename(columns={'id_empleado': 'num_empleados'})
    else:
        empleados_mes = pd.DataFrame(columns=['id_tiempo', 'num_empleados'])

    FACTOR_EMISION = 0.0004
    for _, row in pivot.iterrows():
        id_t = row['id_tiempo']
        ingresos = row.get(1, 0.0)
        costo = row.get(2, 0.0)
        gasto = row.get(3, 0.0)
        consumo = row.get(4, 0.0)
        num_emp = empleados_mes.loc[empleados_mes['id_tiempo'] == id_t, 'num_empleados']
        num_emp = num_emp.iloc[0] if not num_emp.empty else 1

        # Margen Neto (9)
        margen = ((ingresos - costo - gasto) / ingresos * 100) if ingresos > 0 else 0.0
        mediciones.append([id_monitoreo_counter, id_t, 9, 1, margen, 'derivada'])
        id_monitoreo_counter += 1

        # Huella Carbono (10)
        huella = consumo * FACTOR_EMISION
        mediciones.append([id_monitoreo_counter, id_t, 10, 1, huella, 'derivada'])
        id_monitoreo_counter += 1

        # Intensidad Energética (11)
        intensidad = consumo / ingresos if ingresos > 0 else 0.0
        mediciones.append([id_monitoreo_counter, id_t, 11, 1, intensidad, 'derivada'])
        id_monitoreo_counter += 1

        # Productividad (12)
        productividad = ingresos / num_emp if num_emp > 0 else 0.0
        mediciones.append([id_monitoreo_counter, id_t, 12, 1, productividad, 'derivada'])
        id_monitoreo_counter += 1

    print(f"   ✅ Métricas derivadas: Margen Neto, Huella Carbono, Intensidad Energética, Productividad")

   ✅ Métricas derivadas: Margen Neto, Huella Carbono, Intensidad Energética, Productividad


### <font color="lightgreen">Construir fact_monitoreo final</font>

In [21]:
fact_monitoreo = pd.DataFrame(mediciones, columns=['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente'])
fact_monitoreo = fact_monitoreo.sort_values(['id_tiempo', 'id_metrica']).reset_index(drop=True)

print(f"\n✅ fact_monitoreo: {len(fact_monitoreo)} registros")
print(f"   📊 Métricas incluidas: {fact_monitoreo['id_metrica'].nunique()}")
print(f"   📅 Períodos: {fact_monitoreo['id_tiempo'].nunique()}")

hechos['fact_monitoreo'] = fact_monitoreo


✅ fact_monitoreo: 1065 registros
   📊 Métricas incluidas: 12
   📅 Períodos: 96


### <font color="lightgreen">4. Guardar modelo estrella</font>

In [22]:
print("\n" + "="*50)
print("GUARDANDO MODELO ESTRELLA...")
print("="*50)

for nombre, df in dimensiones.items():
    out = f"{DATA_CURATED}/{nombre}.csv"
    df.to_csv(out, index=False, encoding='utf-8')
    print(f"   💾 {nombre}.csv ({len(df)} filas)")

for nombre, df in hechos.items():
    out = f"{DATA_CURATED}/{nombre}.csv"
    df.to_csv(out, index=False, encoding='utf-8')
    print(f"   💾 {nombre}.csv ({len(df)} filas)")


GUARDANDO MODELO ESTRELLA...
   💾 dim_tiempo.csv (2567 filas)
   💾 dim_area.csv (5 filas)
   💾 dim_empleado.csv (25 filas)
   💾 dim_proveedor.csv (5 filas)
   💾 dim_categoria.csv (10 filas)
   💾 dim_metrica.csv (12 filas)
   💾 dim_objetivo.csv (12 filas)
   💾 fact_monitoreo.csv (1065 filas)


### <font color="lightgreen">5. Resumen final</font>

In [23]:
print("\n" + "="*50)
print("✅ MODELO ESTRELLA COMPLETADO")
print("="*50)
print("\n📊 Dimensiones creadas:")
for nombre, df in dimensiones.items():
    print(f"   • {nombre}: {len(df):,} registros")
print("\n📊 Tabla de Hechos:")
for nombre, df in hechos.items():
    print(f"   • {nombre}: {len(df):,} registros, {len(df.columns)} columnas")
print(f"\n📁 Archivos guardados en: {DATA_CURATED}/")
print("\n✅ Listo para conectar al dashboard Streamlit")


✅ MODELO ESTRELLA COMPLETADO

📊 Dimensiones creadas:
   • dim_tiempo: 2,567 registros
   • dim_area: 5 registros
   • dim_empleado: 25 registros
   • dim_proveedor: 5 registros
   • dim_categoria: 10 registros
   • dim_metrica: 12 registros
   • dim_objetivo: 12 registros

📊 Tabla de Hechos:
   • fact_monitoreo: 1,065 registros, 6 columnas

📁 Archivos guardados en: ../data/curated/

✅ Listo para conectar al dashboard Streamlit
